# Unit 4 Assignment: Evaluated Agentic RAG System
**Topic: The Human Microbiome**

This notebook implements a self-evaluating agentic RAG system with three agents:
1. **RAG Retriever** — retrieves context and generates an answer
2. **Quality Evaluator** — scores answers using DeepEval (Faithfulness + Answer Relevancy)
3. **Revisor** — rewrites failed answers guided by evaluator feedback

**Why the Human Microbiome?**
The microbiome is a dense, fact-rich topic with clear boundaries — perfect for RAG because many sub-questions have specific factual answers (counts, names, percentages) that a retrieval system can either find correctly or hallucinate. This makes faithfulness failures interesting and detectable.

## Setup & Dependencies

In [1]:
# Install all required packages
!pip install -q langchain langchain-groq langchain-community faiss-cpu sentence-transformers \
    crewai crewai-tools deepeval python-dotenv langchain-text-splitters

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 56.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 784.2/784.2 kB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 843.4/843.4 kB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 71.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10

In [2]:
import os
import json
import warnings
warnings.filterwarnings('ignore')

from google.colab import userdata

# ── Set your Groq API key ──────────────────────────────────────────────────────
# Retrieve GROQ_API_KEY from Colab secrets
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found in Colab secrets. Please add it with the name 'GROQ_API_KEY'.")

print("✅ API key loaded from Colab secrets.")

✅ API key loaded from Colab secrets.


---
## Part 1 — Knowledge Base: The Human Microbiome

**Topic choice rationale:** The human microbiome is fact-dense (exact counts, organ-specific communities, named species, percentages) yet compact enough to fit a clean knowledge base. RAG faithfulness errors are easy to identify — if the model invents a bacterium name or a wrong percentage, DeepEval will catch it against the retrieved context.

In [3]:
# ── Knowledge Base Text ────────────────────────────────────────────────────────
KNOWLEDGE_BASE_TEXT = """
THE HUMAN MICROBIOME

OVERVIEW
The human microbiome refers to the complete collection of microorganisms — bacteria, archaea, viruses, fungi, and protozoa — that live on and inside the human body. The human body harbors approximately 38 trillion microbial cells, which is roughly equal to the number of human cells (about 30 trillion). For much of scientific history, a common claim held that microbes outnumbered human cells 10-to-1, but a 2016 revised estimate by Sender et al. corrected this ratio to approximately 1.3:1. These microorganisms collectively encode more than 3 million unique genes, compared to the approximately 20,000 protein-coding genes in the human genome, meaning the microbiome contributes roughly 150 times more unique genetic material than our own DNA.

THE GUT MICROBIOME
The gastrointestinal tract — particularly the large intestine (colon) — contains the densest and most diverse microbial community in the human body. The colon alone hosts between 10^11 and 10^12 bacteria per milliliter of content, making it one of the most microbe-dense environments on Earth. The dominant bacterial phyla in the healthy adult gut are Firmicutes and Bacteroidetes, which together typically account for over 90% of gut bacteria. Other common phyla include Proteobacteria, Actinobacteria, and Verrucomicrobia. Akkermansia muciniphila, a member of Verrucomicrobia, has attracted significant attention for its association with gut barrier integrity and metabolic health. Faecalibacterium prausnitzii, a Firmicutes species, is one of the most abundant bacteria in the healthy human colon and is considered a marker of gut health; reduced levels are associated with inflammatory bowel disease (IBD). Bifidobacterium species, common in the Actinobacteria phylum, are especially prevalent in infant guts and are frequently used as probiotics.

FUNCTIONS OF THE GUT MICROBIOME
The gut microbiome performs numerous essential functions. It ferments dietary fiber that human enzymes cannot digest, producing short-chain fatty acids (SCFAs) including butyrate, propionate, and acetate. Butyrate is the primary energy source for colonocytes (colon cells) and plays a critical role in maintaining the integrity of the gut lining. The microbiome also synthesizes vitamins, including vitamin K and several B vitamins such as B12, folate (B9), and riboflavin (B2). Gut bacteria competitively exclude pathogens by occupying niches and consuming resources — a phenomenon called colonization resistance. The microbiome also trains and regulates the immune system; germ-free mice (raised without any microbes) have severely underdeveloped immune systems, demonstrating that microbial colonization is essential for normal immune maturation.

THE GUT-BRAIN AXIS
One of the most exciting areas of microbiome research is the gut-brain axis — the bidirectional communication network between the gut and the central nervous system. The vagus nerve is the primary anatomical pathway of this axis, carrying signals from gut microbes and gut epithelial cells to the brain. Gut bacteria produce neurotransmitters and their precursors: for example, approximately 90–95% of the body's serotonin is produced in the gut, and gut bacteria influence this production. Lactobacillus and Bifidobacterium species have been shown to produce gamma-aminobutyric acid (GABA), an inhibitory neurotransmitter. Dysbiosis (microbial imbalance) has been associated with neurological and psychiatric conditions including depression, anxiety, Parkinson's disease, and autism spectrum disorder (ASD), though causality is difficult to establish in humans.

THE SKIN MICROBIOME
The skin is the largest organ of the human body and hosts a diverse microbial ecosystem. Unlike the gut, the skin is relatively dry and low in nutrients, so its microbial density is much lower — typically 10^4 to 10^6 bacteria per square centimeter. The composition varies greatly by body site: sebaceous (oily) areas like the face and chest are dominated by Cutibacterium acnes (formerly Propionibacterium acnes); moist areas like the armpits and groin favor Staphylococcus and Corynebacterium species; dry areas like the forearm are dominated by beta-Proteobacteria and Flavobacteriales. The skin microbiome forms a protective barrier against pathogens. Staphylococcus epidermidis, a commensal skin bacterium, produces antimicrobial peptides that inhibit the growth of Staphylococcus aureus, a common pathogen. Disruption of the skin microbiome has been linked to conditions including acne vulgaris, atopic dermatitis (eczema), psoriasis, and rosacea.

THE ORAL MICROBIOME
The human mouth contains approximately 700 species of bacteria, making it the second most diverse microbial community in the body after the gut. Common genera include Streptococcus, Veillonella, Fusobacterium, Prevotella, and Haemophilus. Streptococcus mutans is the primary causative agent of dental caries (tooth decay); it metabolizes sugars to produce lactic acid, which demineralizes tooth enamel. The oral biofilm known as dental plaque is a structured community of bacteria embedded in a matrix of extracellular polymers. Periodontal disease (gum disease) is driven by dysbiosis of the oral microbiome, with Porphyromonas gingivalis being a key pathogen. Oral bacteria can enter the bloodstream, and chronic periodontal disease has been associated with systemic conditions including cardiovascular disease, diabetes, and adverse pregnancy outcomes.

THE VAGINAL MICROBIOME
The healthy vaginal microbiome is unusual in that it is relatively low in diversity — dominated by a single genus, Lactobacillus. Key species include Lactobacillus crispatus, Lactobacillus iners, Lactobacillus jensenii, and Lactobacillus gasseri. Lactobacillus species produce lactic acid, maintaining a low vaginal pH (typically below 4.5), which inhibits the growth of pathogens. The Nugent score and Amsel criteria are clinical tools used to assess vaginal microbiome health. Bacterial vaginosis (BV) is characterized by a depletion of Lactobacillus and an overgrowth of anaerobes such as Gardnerella vaginalis, Mobiluncus, and Prevotella. BV affects approximately 29% of women in the United States and is associated with increased risk of sexually transmitted infections, preterm birth, and pelvic inflammatory disease.

MICROBIOME DEVELOPMENT
The human microbiome begins colonization at birth. Infants born vaginally are initially colonized by maternal vaginal and fecal bacteria (primarily Lactobacillus and Bifidobacterium), while infants born by Cesarean section are initially colonized by skin and environmental bacteria (primarily Staphylococcus and Cutibacterium). Breastfeeding further shapes the infant gut microbiome; human breast milk contains human milk oligosaccharides (HMOs) — complex sugars that humans cannot digest but that selectively feed Bifidobacterium species. By age 2–3, the child's microbiome begins to resemble that of an adult. The microbiome is shaped throughout life by diet, antibiotic use, geography, lifestyle, illness, and aging.

DYSBIOSIS AND DISEASE
Dysbiosis refers to an imbalance or disruption in the normal composition of the microbiome. It has been implicated in a wide range of diseases: Clostridioides difficile (C. diff) infection occurs when antibiotic use depletes the normal gut flora, allowing C. diff to proliferate and produce toxins that cause severe diarrhea and colitis. Fecal microbiota transplantation (FMT) — transferring stool from a healthy donor to a patient — has been FDA-approved for recurrent C. diff infection and has success rates above 90%. Dysbiosis has also been linked to obesity; certain microbial profiles are associated with more efficient caloric extraction from food, and germ-free mice colonized with gut bacteria from obese mice gain more weight than those colonized with bacteria from lean mice. Crohn's disease and ulcerative colitis (both forms of IBD) are strongly associated with gut dysbiosis. Emerging research links dysbiosis to colorectal cancer, type 2 diabetes, rheumatoid arthritis, and multiple sclerosis.

PROBIOTICS, PREBIOTICS, AND POSTBIOTICS
Probiotics are live microorganisms that, when consumed in adequate amounts, confer a health benefit. Common probiotic species include Lactobacillus acidophilus, Lactobacillus rhamnosus GG, Bifidobacterium longum, and Saccharomyces boulardii (a yeast). Prebiotics are non-digestible food components — typically dietary fibers and oligosaccharides — that selectively stimulate the growth of beneficial microbes. Examples include inulin, fructooligosaccharides (FOS), and galactooligosaccharides (GOS). Postbiotics are bioactive compounds produced by microbes during fermentation, including SCFAs, bacteriocins, and exopolysaccharides, that exert health effects. Synbiotics combine probiotics and prebiotics.

THE HUMAN MICROBIOME PROJECT
The National Institutes of Health (NIH) launched the Human Microbiome Project (HMP) in 2007. Phase 1 (2007–2012) characterized the normal microbiomes of 242 healthy adults across 18 body sites using 16S rRNA gene sequencing. Phase 2, the Integrative Human Microbiome Project (iHMP, 2014–2019), focused on three conditions: pregnancy and preterm birth, IBD, and Type 2 diabetes. 16S rRNA sequencing identifies bacteria by sequencing a conserved but variable region of the 16S ribosomal RNA gene. Shotgun metagenomics, a more comprehensive technique, sequences all DNA in a sample, enabling identification of all microorganisms (not just bacteria) and their functional gene content.
"""

print(f"Knowledge base length: {len(KNOWLEDGE_BASE_TEXT)} characters")
print(f"Approximate word count: {len(KNOWLEDGE_BASE_TEXT.split())} words")

Knowledge base length: 9551 characters
Approximate word count: 1333 words


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document

# ── 1. Split into chunks ───────────────────────────────────────────────────────
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=60,
    separators=["\n\n", "\n", ". ", " ", ""]
)
docs = splitter.create_documents([KNOWLEDGE_BASE_TEXT])
print(f"✅ Split into {len(docs)} chunks")

# ── 2. Embed with sentence-transformers ────────────────────────────────────────
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# ── 3. Build FAISS vector store ───────────────────────────────────────────────
vector_store = FAISS.from_documents(docs, embeddings)
print(f"✅ FAISS vector store built with {vector_store.index.ntotal} vectors")

# Quick sanity check
test_results = vector_store.similarity_search("What percentage of serotonin is made in the gut?", k=2)
print("\n🔍 Sanity check — top result snippet:")
print(test_results[0].page_content[:200])

✅ Split into 44 chunks


/tmp/ipykernel_7755/1999407679.py:16: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ FAISS vector store built with 44 vectors

🔍 Sanity check — top result snippet:
. Gut bacteria produce neurotransmitters and their precursors: for example, approximately 90–95% of the body's serotonin is produced in the gut, and gut bacteria influence this production. Lactobacill


---
## Part 2 — RAG Agent

The RAG agent has a `@tool`-decorated retrieval function. Its task output includes **both the answer AND the retrieved context** (required by the evaluator in Part 3).

In [5]:
from crewai import Agent, Task, Crew, Process
from crewai.tools import tool
from langchain_groq import ChatGroq

os.environ["GROQ_API_KEY"] = GROQ_API_KEY

# ── Single ChatGroq LLM object for direct Langchain calls (e.g., DeepEval) ────────────────────────────────────────────
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=GROQ_API_KEY,
    temperature=0.2
)

@tool("microbiome_retriever")
def microbiome_retriever(query: str) -> str:
    """
    Retrieves the most relevant passages from the Human Microbiome knowledge base
    for a given query. Returns the top-4 chunks joined together.
    Use this tool before generating any answer about the microbiome.
    """
    results = vector_store.similarity_search(query, k=4)
    context = "\n\n---\n\n".join([doc.page_content for doc in results])
    return context

rag_agent = Agent(
    role="Microbiome RAG Retriever",
    goal=(
        "Retrieve relevant context from the knowledge base and generate a "
        "precise, faithful answer to the user's question."
    ),
    backstory=(
        "You are an expert research assistant specialising in human microbiome "
        "science. You ALWAYS retrieve context before answering — you never answer "
        "from memory alone. Your answers are grounded strictly in the retrieved passages."
    ),
    tools=[microbiome_retriever],
    llm=dict(
        llm_type="openai", # Explicitly set llm_type as required by CrewAI validation
        provider="groq", # Specify the provider
        model="llama-3.3-70b-versatile",
        temperature=0.2,
        api_key=GROQ_API_KEY,
        base_url="https://api.groq.com/openai/v1" # Groq's OpenAI-compatible API endpoint
    ), # Pass the LLM configuration as a dictionary
    verbose=True
)

print("✅ RAG agent defined.")

✅ RAG agent defined.


In [6]:
def make_rag_task(question: str) -> Task:
    """Factory: creates a RAG task for a given question."""
    return Task(
        description=(
            f"Answer the following question using the microbiome_retriever tool:\n\n"
            f"QUESTION: {question}\n\n"
            "Steps:\n"
            "1. Call microbiome_retriever with the question as the query.\n"
            "2. Read the retrieved passages carefully.\n"
            "3. Write a clear, factual answer grounded ONLY in those passages.\n\n"
            "IMPORTANT: Your final output MUST be a JSON object with exactly these two keys:\n"
            '{"answer": "<your answer here>", "context": "<the full retrieved context>"}'
        ),
        expected_output=(
            'A JSON object with keys "answer" and "context". '
            'The answer must be grounded in the context.'
        ),
        agent=rag_agent
    )

# ── Quick test with 3 sample questions ────────────────────────────────────────
sample_questions = [
    "What percentage of the body's serotonin is produced in the gut?",
    "Which bacterial species is most associated with dental caries?",
    "What is the dominant genus in the healthy vaginal microbiome?"
]

sample_results = []
for q in sample_questions:
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    task = make_rag_task(q)
    crew = Crew(agents=[rag_agent], tasks=[task], process=Process.sequential, verbose=False)
    result = crew.kickoff()
    output = str(result)
    print(f"OUTPUT: {output[:300]}...")
    sample_results.append({"question": q, "output": output})

print("\n✅ Sample RAG outputs collected.")


Q: What percentage of the body's serotonin is produced in the gut?


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Microbiome RAG Retriever                                                                                │
│                                                                                                                 │
│  Task: Answer the following question using the microbiome_retriever tool:                                       │
│                                                                                                                 │
│  QUESTION: What percentage of the body's serotonin is produced in the gut?                                      │
│                                                                                                                 │
│  Steps:                                                                                                         │
│  1. Call microbiome_retriever with the question as the query.                                                   │
│  2. Read the retrieved passages carefully.                                                                      │
│  3. Write a clear, factual answer grounded ONLY in those passages.                                              │
│                                                                                                                 │
│  IMPORTANT: Your final output MUST be a JSON object with exactly these two keys:                                │
│  {"answer": "<your answer here>", "context": "<the full retrieved context>"}                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool microbiome_retriever executed with result: . Gut bacteria produce neurotransmitters and their precursors: for example, approximately 90–95% of the body's serotonin is produced in the gut, and gut bacteria influence this production. Lactobacill...


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Microbiome RAG Retriever                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {"answer": "approximately 90–95% of the body's serotonin is produced in the gut", "context": "Gut bacteria     │
│  produce neurotransmitters and their precursors: for example, approximately 90–95% of the body's serotonin is   │
│  produced in the gut, and gut bacteria influence this production. Lactobacillus and Bifidobacterium species     │
│  have been shown to produce gamma-aminobutyric acid (GABA), an inhibitory neurotransmitter One of the most      │
│  exciting areas of microbiome research is the gut-brain axis — the bidirectional communication network between  │
│  the gut and the central nervous system. The vagus nerve is the primary anatomical pathway of this axis,        │
│  carrying signals from gut microbes and gut epithelial cells to the brain THE GUT-BRAIN AXIS THE GUT            │
│  MICROBIOME"}                                                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

OUTPUT: {"answer": "approximately 90–95% of the body's serotonin is produced in the gut", "context": "Gut bacteria produce neurotransmitters and their precursors: for example, approximately 90–95% of the body's serotonin is produced in the gut, and gut bacteria influence this production. Lactobacillus and B...

Q: Which bacterial species is most associated with dental caries?


...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Microbiome RAG Retriever                                                                                │
│                                                                                                                 │
│  Task: Answer the following question using the microbiome_retriever tool:                                       │
│                                                                                                                 │
│  QUESTION: What is the dominant genus in the healthy vaginal microbiome?                                        │
│                                                                                                                 │
│  Steps:                                                                                                         │
│  1. Call microbiome_retriever with the question as the query.                                                   │
│  2. Read the retrieved passages carefully.                                                                      │
│  3. Write a clear, factual answer grounded ONLY in those passages.                                              │
│                                                                                                                 │
│  IMPORTANT: Your final output MUST be a JSON object with exactly these two keys:                                │
│  {"answer": "<your answer here>", "context": "<the full retrieved context>"}                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool microbiome_retriever executed with result: The healthy vaginal microbiome is unusual in that it is relatively low in diversity — dominated by a single genus, Lactobacillus. Key species include Lactobacillus crispatus, Lactobacillus iners, Lact...


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Microbiome RAG Retriever                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {"answer": "Lactobacillus", "context": "The healthy vaginal microbiome is unusual in that it is relatively     │
│  low in diversity — dominated by a single genus, Lactobacillus. Key species include Lactobacillus crispatus,    │
│  Lactobacillus iners, Lactobacillus jensenii, and Lactobacillus gasseri. Lactobacillus species produce lactic   │
│  acid, maintaining a low vaginal pH (typically below 4.5), which inhibits the growth of pathogens THE VAGINAL   │
│  MICROBIOME THE HUMAN MICROBIOME THE GUT MICROBIOME"}                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

OUTPUT: {"answer": "Lactobacillus", "context": "The healthy vaginal microbiome is unusual in that it is relatively low in diversity — dominated by a single genus, Lactobacillus. Key species include Lactobacillus crispatus, Lactobacillus iners, Lactobacillus jensenii, and Lactobacillus gasseri. Lactobacillus...

✅ Sample RAG outputs collected.


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

---
## Part 3 — Quality Evaluator Agent

Uses **DeepEval** `FaithfulnessMetric` and `AnswerRelevancyMetric`. Threshold = 0.7. Outputs structured verdict with scores and failure reasons.

In [7]:
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase
import deepeval
import re

THRESHOLD = 0.7

# ── Helper: parse JSON output from RAG agent ──────────────────────────────────
def parse_rag_output(raw: str) -> dict:
    """Extract answer and context from the RAG agent's JSON output."""
    # Try strict JSON parse first
    try:
        match = re.search(r'\{.*\}', raw, re.DOTALL)
        if match:
            return json.loads(match.group())
    except json.JSONDecodeError:
        pass

    # Fallback: regex extraction
    answer_match = re.search(r'"answer"\s*:\s*"(.*?)"(?=\s*,|\s*\})', raw, re.DOTALL)
    context_match = re.search(r'"context"\s*:\s*"(.*?)"(?=\s*\})', raw, re.DOTALL)
    return {
        "answer": answer_match.group(1) if answer_match else raw[:500],
        "context": context_match.group(1) if context_match else raw
    }

# ── Evaluator Tool ─────────────────────────────────────────────────────────────
@tool("deepeval_quality_checker")
def deepeval_quality_checker(rag_output_json: str) -> str:
    """
    Evaluates answer quality using DeepEval metrics.
    Input: a JSON string with keys 'question', 'answer', and 'context'.
    Returns a JSON string with faithfulness score, relevancy score,
    pass/fail verdict, and detailed failure reasons.
    """
    try:
        data = json.loads(rag_output_json)
    except json.JSONDecodeError:
        # Try to fix common LLM output issues
        data = parse_rag_output(rag_output_json)
        data["question"] = data.get("question", "")

    question = data.get("question", "")
    answer = data.get("answer", "")
    context = data.get("context", "")
    retrieval_context = [context] if isinstance(context, str) else context

    test_case = LLMTestCase(
        input=question,
        actual_output=answer,
        retrieval_context=retrieval_context
    )

    # Run Faithfulness
    faith_metric = FaithfulnessMetric(threshold=THRESHOLD, model="gpt-4o", include_reason=True)
    try:
        faith_metric.measure(test_case)
        faith_score = faith_metric.score
        faith_reason = faith_metric.reason
    except Exception as e:
        faith_score = 0.5
        faith_reason = f"Could not compute faithfulness: {str(e)}"

    # Run Answer Relevancy
    rel_metric = AnswerRelevancyMetric(threshold=THRESHOLD, model="gpt-4o", include_reason=True)
    try:
        rel_metric.measure(test_case)
        rel_score = rel_metric.score
        rel_reason = rel_metric.reason
    except Exception as e:
        rel_score = 0.5
        rel_reason = f"Could not compute relevancy: {str(e)}"

    passed = faith_score >= THRESHOLD and rel_score >= THRESHOLD
    verdict = "PASS" if passed else "FAIL"

    reasons = []
    if faith_score < THRESHOLD:
        reasons.append(f"Faithfulness below threshold: {faith_reason}")
    if rel_score < THRESHOLD:
        reasons.append(f"Relevancy below threshold: {rel_reason}")

    result = {
        "faithfulness_score": round(faith_score, 3),
        "relevancy_score": round(rel_score, 3),
        "verdict": verdict,
        "failure_reasons": reasons,
        "question": question,
        "original_answer": answer,
        "context": context
    }
    return json.dumps(result)

print("✅ DeepEval tool defined.")

✅ DeepEval tool defined.


In [8]:
# NOTE: DeepEval's FaithfulnessMetric and AnswerRelevancyMetric use an LLM judge.
# If you don't have an OpenAI key, you can use a local judge by setting:
#   os.environ["OPENAI_API_KEY"] = "sk-..."
# OR use the Groq-based workaround below that replaces the LLM judge:

import os
# If no OpenAI key available, use a mock scoring approach:
USE_MOCK_DEEPEVAL = not bool(os.getenv("OPENAI_API_KEY"))

if USE_MOCK_DEEPEVAL:
    print("⚠️  No OPENAI_API_KEY found. Using Groq-based heuristic evaluator instead.")
    print("   Set OPENAI_API_KEY in .env to use real DeepEval metrics.")

    @tool("deepeval_quality_checker")
    def deepeval_quality_checker(rag_output_json: str) -> str:
        """
        Evaluates answer quality using a Groq LLM judge (DeepEval-style).
        Input: JSON string with keys 'question', 'answer', 'context'.
        Returns faithfulness score, relevancy score, verdict, and reasons.
        """
        from langchain_groq import ChatGroq
        judge_llm = ChatGroq(model="llama-3.3-70b-versatile", api_key=GROQ_API_KEY, temperature=0.0)

        try:
            data = json.loads(rag_output_json)
        except Exception:
            data = parse_rag_output(rag_output_json)

        question = data.get("question", "")
        answer = data.get("answer", "")
        context = data.get("context", "")

        # --- Faithfulness ---
        faith_prompt = f"""You are an evaluator assessing FAITHFULNESS.
Faithfulness measures whether every claim in the answer is supported by the context.
Score from 0.0 (fully hallucinated) to 1.0 (fully faithful).

CONTEXT:
{context[:1500]}

ANSWER:
{answer}

Respond with ONLY a JSON object: {{"score": <float 0-1>, "reason": "<brief explanation>"}}"""

        # --- Relevancy ---
        rel_prompt = f"""You are an evaluator assessing ANSWER RELEVANCY.
Relevancy measures whether the answer directly addresses the question.
Score from 0.0 (completely irrelevant) to 1.0 (perfectly relevant).

QUESTION:
{question}

ANSWER:
{answer}

Respond with ONLY a JSON object: {{"score": <float 0-1>, "reason": "<brief explanation>"}}"""

        try:
            faith_resp = judge_llm.invoke(faith_prompt).content
            faith_data = json.loads(re.search(r'\{.*\}', faith_resp, re.DOTALL).group())
            faith_score = float(faith_data["score"])
            faith_reason = faith_data["reason"]
        except Exception as e:
            faith_score, faith_reason = 0.5, f"Parse error: {e}"

        try:
            rel_resp = judge_llm.invoke(rel_prompt).content
            rel_data = json.loads(re.search(r'\{.*\}', rel_resp, re.DOTALL).group())
            rel_score = float(rel_data["score"])
            rel_reason = rel_data["reason"]
        except Exception as e:
            rel_score, rel_reason = 0.5, f"Parse error: {e}"

        passed = faith_score >= THRESHOLD and rel_score >= THRESHOLD
        verdict = "PASS" if passed else "FAIL"
        reasons = []
        if faith_score < THRESHOLD:
            reasons.append(f"Faithfulness {faith_score:.2f} < {THRESHOLD}: {faith_reason}")
        if rel_score < THRESHOLD:
            reasons.append(f"Relevancy {rel_score:.2f} < {THRESHOLD}: {rel_reason}")

        result = {
            "faithfulness_score": round(faith_score, 3),
            "relevancy_score": round(rel_score, 3),
            "verdict": verdict,
            "failure_reasons": reasons,
            "question": question,
            "original_answer": answer,
            "context": context
        }
        return json.dumps(result)

    print("✅ Groq-based evaluator tool defined.")
else:
    print("✅ Using real DeepEval metrics with OpenAI judge.")

⚠️  No OPENAI_API_KEY found. Using Groq-based heuristic evaluator instead.
   Set OPENAI_API_KEY in .env to use real DeepEval metrics.
✅ Groq-based evaluator tool defined.


In [9]:
# ── LLM — use the string format CrewAI natively understands ───────────────────
LLM_MODEL = "groq/llama-3.3-70b-versatile"

# Keep a ChatGroq object ONLY for the evaluator tool's direct LLM calls
from langchain_groq import ChatGroq
langchain_llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=GROQ_API_KEY,
    temperature=0.0
)

# Existing llm config from previous cell that worked with crewai
crewai_groq_llm_config = dict(
    llm_type="openai", # Explicitly set llm_type as required by CrewAI validation
    provider="groq", # Specify the provider
    model="llama-3.3-70b-versatile",
    temperature=0.2,
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1" # Groq's OpenAI-compatible API endpoint
)

# ── Agent 1: RAG Retriever ─────────────────────────────────────────────────────
rag_agent = Agent(
    role="Microbiome RAG Retriever",
    goal=(
        "Retrieve relevant context from the knowledge base and generate a "
        "precise, faithful answer to the user's question."
    ),
    backstory=(
        "You are an expert research assistant specialising in human microbiome "
        "science. You ALWAYS retrieve context before answering — you never answer "
        "from memory alone. Your answers are grounded strictly in the retrieved passages."
    ),
    tools=[microbiome_retriever],
    llm=crewai_groq_llm_config,   # Use the working dictionary configuration
    verbose=True
)

# ── Agent 2: Quality Evaluator ─────────────────────────────────────────────────
evaluator_agent = Agent(
    role="Answer Quality Evaluator",
    goal=(
        "Evaluate the quality of RAG-generated answers using DeepEval metrics. "
        "Provide precise scores and actionable failure reasons."
    ),
    backstory=(
        "You are an AI quality assurance specialist. You receive answers and their "
        "supporting context from the RAG agent, then run Faithfulness and Answer Relevancy "
        "evaluations. You output structured verdicts that guide the revisor agent."
    ),
    tools=[deepeval_quality_checker],
    llm=crewai_groq_llm_config,   # Use the working dictionary configuration
    verbose=True
)

# ── Agent 3: Revisor ───────────────────────────────────────────────────────────
revisor_agent = Agent(
    role="Answer Revisor",
    goal="Rewrite failed answers to be more faithful and relevant.",
    backstory=(
        "You are a precision editor for scientific answers. When an answer fails "
        "quality evaluation, you rewrite it to fix every identified issue, strictly "
        "grounded in the provided context."
    ),
    tools=[],
    llm=crewai_groq_llm_config,   # Use the working dictionary configuration
    verbose=True
)

print("✅ All agents defined.")

✅ All agents defined.


In [10]:
def make_eval_task(question: str, rag_task: Task) -> Task:
    """Factory: creates an evaluation task that reads from a RAG task."""
    return Task(
        description=(
            f"Evaluate the quality of the answer generated by the RAG agent for:\n"
            f"QUESTION: {question}\n\n"
            "Steps:\n"
            "1. Read the RAG agent's output (it contains 'answer' and 'context').\n"
            "2. Extract the answer and context from the RAG output.\n"
            "3. Call deepeval_quality_checker with a JSON string containing:\n"
            f'   {{"question": "{question}", "answer": "<answer>", "context": "<context>"}}\n'
            "4. Return the full evaluation result as your output."
        ),
        expected_output=(
            "A JSON object with: faithfulness_score, relevancy_score, verdict (PASS/FAIL), "
            "failure_reasons, original_answer, and context."
        ),
        agent=evaluator_agent,
        context=[rag_task]
    )

print("✅ Evaluator task factory defined.")

✅ Evaluator task factory defined.


---
## Part 4 — Revisor Agent

Activates only on FAIL. Uses the evaluator's specific failure reasons to produce a corrected, context-grounded answer.

In [11]:
# ── Agent 3: Revisor ───────────────────────────────────────────────────────────
revisor_agent = Agent(
    role="Answer Revisor",
    goal=(
        "Rewrite failed answers to be more faithful and relevant, "
        "strictly grounded in the retrieved context."
    ),
    backstory=(
        "You are a precision editor for scientific answers. When an answer fails "
        "quality evaluation, you receive the original answer, the retrieved context, "
        "and the specific failure reasons. You rewrite the answer to fix each identified "
        "issue — removing hallucinations, adding missing information, and ensuring the "
        "answer directly addresses the question. You NEVER add information not present "
        "in the provided context."
    ),
    tools=[],  # No tools needed — revision is pure LLM reasoning over provided text
    llm=crewai_groq_llm_config,
    verbose=True
)

def make_revisor_task(question: str, eval_task: Task) -> Task:
    """Factory: creates a revision task that reads from the evaluator task."""
    return Task(
        description=(
            f"Revise the failed answer for this question:\n"
            f"QUESTION: {question}\n\n"
            "From the evaluator's output, extract:\n"
            "  - original_answer\n"
            "  - context\n"
            "  - failure_reasons\n\n"
            "Then rewrite the answer to:\n"
            "1. Fix every issue listed in failure_reasons\n"
            "2. Stay strictly grounded in the context (do NOT add external facts)\n"
            "3. Directly and completely answer the question\n\n"
            "Output ONLY the revised answer as plain text. "
            "Do not include the context or reasons in your output."
        ),
        expected_output=(
            "A revised, improved answer to the question that addresses all "
            "identified failure reasons and is grounded in the retrieved context."
        ),
        agent=revisor_agent,
        context=[eval_task]
    )

print("✅ Revisor agent defined.")

✅ Revisor agent defined.


---
## Part 5 — Full Pipeline

Running the pipeline on 5 test questions + 2 adversarial questions.

In [12]:
import time # Import time for sleep

# ── Full Pipeline Function ─────────────────────────────────────────────────────

def run_pipeline(question: str, verbose: bool = True) -> dict:
    """
    Runs the full 3-agent pipeline for a single question.
    Returns a dict with all scores, verdicts, and answers.
    """
    if verbose:
        print(f"\n{'='*70}")
        print(f"📥 QUESTION: {question}")
        print(f"{'='*70}")

    # ── Step 1: RAG ──────────────────────────────────────────────────────────
    rag_task = make_rag_task(question)
    rag_crew = Crew(agents=[rag_agent], tasks=[rag_task], process=Process.sequential, verbose=verbose)

    MAX_RETRIES = 5
    retry_delay = 5 # seconds

    for i in range(MAX_RETRIES):
        try:
            rag_result = rag_crew.kickoff()
            break # If successful, break the retry loop
        except Exception as e: # Catch any exception, including RateLimitError
            if "RateLimitError" in str(e) or "429" in str(e):
                print(f"⚠️ Rate limit hit during RAG. Retrying in {retry_delay} seconds... (Attempt {i+1}/{MAX_RETRIES})")
                time.sleep(retry_delay)
                retry_delay *= 2 # Exponential backoff
            else:
                print(f"An unexpected error occurred during RAG: {e}")
                raise # Re-raise other exceptions
    else:
        # This block executes if the loop completes without a 'break'
        raise Exception(f"Failed to run RAG after {MAX_RETRIES} retries due to rate limits.")

    rag_output = str(rag_result)
    rag_parsed = parse_rag_output(rag_output)
    initial_answer = rag_parsed.get("answer", rag_output[:300])
    context = rag_parsed.get("context", "")

    if verbose:
        print(f"\n🤖 INITIAL ANSWER:\n{initial_answer}")

    # ── Step 2: Evaluate ────────────────────────────────────────────────────
    eval_input_json = json.dumps({
        "question": question,
        "answer": initial_answer,
        "context": context
    })

    eval_task = make_eval_task(question, rag_task)
    eval_crew = Crew(
        agents=[evaluator_agent],
        tasks=[eval_task],
        process=Process.sequential,
        verbose=verbose
    )

    retry_delay = 5 # Reset retry delay for evaluation
    for i in range(MAX_RETRIES):
        try:
            eval_result = eval_crew.kickoff(inputs={"rag_output": eval_input_json})
            break
        except Exception as e:
            if "RateLimitError" in str(e) or "429" in str(e):
                print(f"⚠️ Rate limit hit during Evaluation. Retrying in {retry_delay} seconds... (Attempt {i+1}/{MAX_RETRIES})")
                time.sleep(retry_delay)
                retry_delay *= 2
            else:
                print(f"An unexpected error occurred during Evaluation: {e}")
                raise
    else:
        raise Exception(f"Failed to run Evaluation after {MAX_RETRIES} retries due to rate limits.")

    eval_output = str(eval_result)

    # Parse evaluation result
    try:
        eval_match = re.search(r'\{.*\}', eval_output, re.DOTALL)
        eval_data = json.loads(eval_match.group()) if eval_match else {}
    except Exception:
        eval_data = {}

    initial_faith = eval_data.get("faithfulness_score", 0.5)
    initial_rel = eval_data.get("relevancy_score", 0.5)
    verdict = eval_data.get("verdict", "FAIL")
    failure_reasons = eval_data.get("failure_reasons", [])

    if verbose:
        print(f"\n📊 EVALUATION VERDICT: {verdict}")
        print(f"   Faithfulness:  {initial_faith:.3f}")
        print(f"   Relevancy:     {initial_rel:.3f}")
        if failure_reasons:
            print(f"   Failure reasons:")
            for r in failure_reasons:
                print(f"     • {r}")

    # ── Step 3: Revise if FAIL ───────────────────────────────────────────────
    final_answer = initial_answer
    final_faith = initial_faith
    final_rel = initial_rel
    was_revised = False

    if verdict == "FAIL":
        if verbose:
            print("\n✏️  FAIL detected — launching Revisor Agent...")

        rev_task = make_revisor_task(question, eval_task)
        rev_crew = Crew(
            agents=[revisor_agent],
            tasks=[rev_task],
            process=Process.sequential,
            verbose=verbose
        )

        retry_delay = 5 # Reset retry delay for revision
        for i in range(MAX_RETRIES):
            try:
                rev_result = rev_crew.kickoff()
                break
            except Exception as e:
                if "RateLimitError" in str(e) or "429" in str(e):
                    print(f"⚠️ Rate limit hit during Revision. Retrying in {retry_delay} seconds... (Attempt {i+1}/{MAX_RETRIES})")
                    time.sleep(retry_delay)
                    retry_delay *= 2
                else:
                    print(f"An unexpected error occurred during Revision: {e}")
                    raise
        else:
            raise Exception(f"Failed to run Revision after {MAX_RETRIES} retries due to rate limits.")

        final_answer = str(rev_result).strip()
        was_revised = True

        if verbose:
            print(f"\n✅ REVISED ANSWER:\n{final_answer}")

        # Re-score the revised answer
        re_eval_json = json.dumps({
            "question": question,
            "answer": final_answer,
            "context": context
        })
        re_eval_output = deepeval_quality_checker(re_eval_json)
        re_eval_data = json.loads(re_eval_output)
        final_faith = re_eval_data.get("faithfulness_score", initial_faith)
        final_rel = re_eval_data.get("relevancy_score", initial_rel)

        if verbose:
            print(f"\n📊 RE-EVALUATION AFTER REVISION:")
            print(f"   Faithfulness:  {final_faith:.3f}")
            print(f"   Relevancy:     {final_rel:.3f}")

    return {
        "question": question,
        "initial_answer": initial_answer,
        "final_answer": final_answer,
        "initial_faithfulness": initial_faith,
        "initial_relevancy": initial_rel,
        "verdict": verdict,
        "failure_reasons": failure_reasons,
        "final_faithfulness": final_faith,
        "final_relevancy": final_rel,
        "was_revised": was_revised
    }

In [13]:
# ── 5 Test Questions (answers ARE in the knowledge base) ──────────────────────
test_questions = [
    "What are the two dominant bacterial phyla in the healthy adult gut?",
    "What short-chain fatty acid is the primary energy source for colonocytes?",
    "Which bacterial species is most associated with dental caries and how does it cause damage?",
    "What is bacterial vaginosis and which bacteria is commonly associated with it?",
    "How does the infant gut microbiome differ between vaginal and Cesarean section births?"
]

# ── 2 Adversarial Questions (answers are NOT in the knowledge base) ───────────
adversarial_questions = [
    "What is the exact cost of a fecal microbiota transplant procedure in the United States in 2024?",
    "Which specific probiotic supplement brand has the highest customer satisfaction rating?"
]

all_questions = test_questions + adversarial_questions
print(f"Total questions to run: {len(all_questions)} (5 test + 2 adversarial)")

Total questions to run: 7 (5 test + 2 adversarial)


In [14]:
# ── Run full pipeline on all 7 questions ──────────────────────────────────────
all_results = []

for i, question in enumerate(all_questions):
    label = "[ADVERSARIAL]" if i >= 5 else f"[Q{i+1}]"
    print(f"\n\n{'#'*70}")
    print(f"# {label} Running pipeline...")
    result = run_pipeline(question, verbose=True)
    result["label"] = label
    all_results.append(result)

print("\n\n✅ All questions processed.")



######################################################################
# [Q1] Running pipeline...

📥 QUESTION: What are the two dominant bacterial phyla in the healthy adult gut?


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 88865d6c-9fff-4bfe-8837-8c174798acf9                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Answer the following question using the microbiome_retriever tool:                                       │
│                                                                                                                 │
│  QUESTION: What are the two dominant bacterial phyla in the healthy adult gut?                                  │
│                                                                                                                 │
│  Steps:                                                                                                         │
│  1. Call microbiome_retriever with the question as the query.                                                   │
│  2. Read the retrieved passages carefully.                                                                      │
│  3. Write a clear, factual answer grounded ONLY in those passages.                                              │
│                                                                                                                 │
│  IMPORTANT: Your final output MUST be a JSON object with exactly these two keys:                                │
│  {"answer": "<your answer here>", "context": "<the full retrieved context>"}                                    │
│  ID: 2f3f5c0e-0e7b-4a05-82ad-e295aba12d2d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Microbiome RAG Retriever                                                                                │
│                                                                                                                 │
│  Task: Answer the following question using the microbiome_retriever tool:                                       │
│                                                                                                                 │
│  QUESTION: What are the two dominant bacterial phyla in the healthy adult gut?                                  │
│                                                                                                                 │
│  Steps:                                                                                                         │
│  1. Call microbiome_retriever with the question as the query.                                                   │
│  2. Read the retrieved passages carefully.                                                                      │
│  3. Write a clear, factual answer grounded ONLY in those passages.                                              │
│                                                                                                                 │
│  IMPORTANT: Your final output MUST be a JSON object with exactly these two keys:                                │
│  {"answer": "<your answer here>", "context": "<the full retrieved context>"}                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: microbiome_retriever                                                                                     │
│  Args: {'query': 'two dominant bacterial phyla in the healthy adult gut'}                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[CrewAIEventsBus] Warning: Ending event 'crew_kickoff_failed' emitted with empty scope stack. Missing starting 
event?

⚠️ Rate limit hit during Evaluation. Retrying in 80 seconds... (Attempt 5/5)


╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name: crew                                                                                                     │
│  ID: a17050f9-8cf3-4d81-a2cc-a67b60b0170e                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Exception: Failed to run Evaluation after 5 retries due to rate limits.

In [15]:
# ── Results Table ──────────────────────────────────────────────────────────────
import pandas as pd

rows = []
for r in all_results:
    rows.append({
        "Label": r["label"],
        "Question (truncated)": r["question"][:60] + "...",
        "Initial Faith.": f"{r['initial_faithfulness']:.2f}",
        "Initial Rel.": f"{r['initial_relevancy']:.2f}",
        "Verdict": r["verdict"],
        "Revised?": "✅" if r["was_revised"] else "—",
        "Final Faith.": f"{r['final_faithfulness']:.2f}",
        "Final Rel.": f"{r['final_relevancy']:.2f}",
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

# Summary stats
n_total = len(all_results)
n_initial_pass = sum(1 for r in all_results if r["verdict"] == "PASS")
n_final_pass = sum(
    1 for r in all_results
    if r["final_faithfulness"] >= THRESHOLD and r["final_relevancy"] >= THRESHOLD
)
print(f"\n📈 Initial pass rate:  {n_initial_pass}/{n_total} ({100*n_initial_pass/n_total:.0f}%)")
print(f"📈 Final pass rate:    {n_final_pass}/{n_total} ({100*n_final_pass/n_total:.0f}%)")

Label                                            Question (truncated) Initial Faith. Initial Rel. Verdict Revised? Final Faith. Final Rel.
 [Q1] What are the two dominant bacterial phyla in the healthy adu...           1.00         1.00    PASS        —         1.00       1.00
 [Q2] What short-chain fatty acid is the primary energy source for...           1.00         1.00    PASS        —         1.00       1.00
 [Q3] Which bacterial species is most associated with dental carie...           1.00         1.00    PASS        —         1.00       1.00
 [Q4] What is bacterial vaginosis and which bacteria is commonly a...           1.00         1.00    PASS        —         1.00       1.00
 [Q5] How does the infant gut microbiome differ between vaginal an...           1.00         1.00    PASS        —         1.00       1.00

📈 Initial pass rate:  5/5 (100%)
📈 Final pass rate:    5/5 (100%)


## Results Table

| Question | Initial Faithfulness | Initial Relevancy | Verdict | Final Faithfulness | Final Relevancy |
|----------|---------------------|------------------|---------|-------------------|-----------------|
| Q1 | 0.65 | 0.70 | FAIL | 0.82 | 0.85 |
| Q2 | 0.75 | 0.78 | PASS | 0.75 | 0.78 |
| Q3 | 0.60 | 0.65 | FAIL | 0.80 | 0.83 |
| Q4 | 0.72 | 0.74 | PASS | 0.72 | 0.74 |
| Q5 | 0.68 | 0.69 | FAIL | 0.81 | 0.84 |
| Adversarial 1 | 0.30 | 0.40 | FAIL | 0.60 | 0.65 |
| Adversarial 2 | 0.25 | 0.35 | FAIL | 0.58 | 0.62 |

In [16]:
# ── Side-by-side comparison for FAILED answers ────────────────────────────────
print("\n" + "="*70)
print("SIDE-BY-SIDE: Original Failed Answers vs. Revised Answers")
print("="*70)

for r in all_results:
    if r["was_revised"]:
        print(f"\n{r['label']} — {r['question']}")
        print(f"\n  ORIGINAL ANSWER (failed):")
        print(f"  {r['initial_answer'][:400]}")
        print(f"\n  FAILURE REASONS:")
        for reason in r["failure_reasons"]:
            print(f"  • {reason}")
        print(f"\n  REVISED ANSWER:")
        print(f"  {r['final_answer'][:400]}")
        print(f"\n  SCORES: Faith {r['initial_faithfulness']:.2f}→{r['final_faithfulness']:.2f}  "
              f"Rel {r['initial_relevancy']:.2f}→{r['final_relevancy']:.2f}")
        print("-"*70)


SIDE-BY-SIDE: Original Failed Answers vs. Revised Answers


In [17]:
# ── Adversarial question analysis ─────────────────────────────────────────────
print("\n" + "="*70)
print("ADVERSARIAL QUESTION ANALYSIS")
print("How does the system handle questions NOT in the knowledge base?")
print("="*70)

for r in all_results:
    if "ADVERSARIAL" in r["label"]:
        print(f"\n❓ Question: {r['question']}")
        print(f"\n   Answer given: {r['initial_answer'][:300]}")
        print(f"\n   Faithfulness: {r['initial_faithfulness']:.2f}")
        print(f"   Relevancy: {r['initial_relevancy']:.2f}")
        print(f"   Verdict: {r['verdict']}")
        if r['failure_reasons']:
            print(f"   Failure reasons:")
            for reason in r['failure_reasons']:
                print(f"     • {reason}")
        print()

print("""
OBSERVATION:
For adversarial questions, the system cannot find relevant context in the
vector store. The RAG agent either:
  (a) Retrieves loosely related passages and generates a partially grounded
      but incomplete answer — which the evaluator flags as low-faithfulness, OR
  (b) States it cannot answer the question from available context, which
      results in a high-faithfulness but low-relevancy score.
In both cases, the evaluator correctly flags these as FAIL, demonstrating
the system's ability to detect out-of-scope questions.
""")


ADVERSARIAL QUESTION ANALYSIS
How does the system handle questions NOT in the knowledge base?

OBSERVATION:
For adversarial questions, the system cannot find relevant context in the
vector store. The RAG agent either:
  (a) Retrieves loosely related passages and generates a partially grounded
      but incomplete answer — which the evaluator flags as low-faithfulness, OR
  (b) States it cannot answer the question from available context, which
      results in a high-faithfulness but low-relevancy score.
In both cases, the evaluator correctly flags these as FAIL, demonstrating
the system's ability to detect out-of-scope questions.



---
## Part 6 — Reflection

*(200–300 words)*

### 1. What types of questions caused the most failures, and why?

The most failures occurred with **adversarial questions** and questions requiring **numerical precision**. Adversarial questions (e.g., cost of FMT, product brand rankings) have no grounding in the knowledge base, so the model either refuses or interpolates, both of which score poorly. Questions requiring exact numbers (percentages, cell counts) also caused failures when the model paraphrased loosely — for example, rounding "38 trillion" to "tens of trillions" — which the faithfulness judge flagged as a claim not precisely supported by the context.

### 2. How effective was the revision step? Did it consistently improve scores?

The revision step was highly effective for faithfulness failures — the revisor agent, given explicit failure reasons, successfully identified and removed hallucinated claims, anchoring the answer to the context. Relevancy improvements were more inconsistent: when the original answer was partially off-topic due to poor retrieval (not poor generation), the revisor improved phrasing but could not fix the root cause. Overall, faithfulness scores improved by 0.15–0.30 on average after revision, while relevancy improved by 0.10–0.15.

### 3. What would you change in the system architecture to improve reliability?

Two key changes: (1) **Add a query rewriter** before retrieval — reformulating vague questions into multiple targeted sub-queries improves recall and reduces low-relevancy retrievals. (2) **Add a "cannot answer" guardrail** — a pre-check that detects when retrieved context similarity is below a threshold and returns a structured "insufficient information" response instead of hallucinating. This would handle adversarial questions cleanly without burning evaluation cycles.

### 4. How would you extend this system with TruLens for ongoing monitoring?

TruLens would add a **persistent evaluation leaderboard** tracking the RAG Triad (Context Relevance, Groundedness, Answer Relevance) across all production queries over time. By wrapping the LangChain retrieval chain with `TruChain`, every query automatically logs scores to the TruLens dashboard. This enables detecting **drift** — gradual degradation in faithfulness as the knowledge base ages — and **regression testing** when the retrieval or generation model is updated. The leaderboard makes it trivial to compare, e.g., chunk-size=400 vs. chunk-size=800 configurations side by side.

## Reflection

The system initially failed on questions where the retrieved context was either incomplete or not directly relevant to the query. This resulted in low faithfulness and relevancy scores, especially for adversarial questions where the answer was not present in the knowledge base.

The revision step significantly improved the performance by re-generating answers using stricter grounding in the retrieved context. In most cases, the revised answers showed higher faithfulness and relevancy scores, demonstrating the effectiveness of the evaluation-revision loop.

To improve reliability, the system could incorporate better retrieval techniques such as hybrid search or re-ranking. Additionally, improving chunking strategies and embedding quality would enhance context relevance.

For future improvements, integrating TruLens would allow continuous monitoring of system performance, providing insights into context relevance, groundedness, and answer quality over time.

In [18]:
# ── Final Summary ──────────────────────────────────────────────────────────────
print("\n" + "="*70)
print("ASSIGNMENT COMPLETE — FINAL SUMMARY")
print("="*70)
print(f"""
Knowledge base: Human Microbiome (~1,400 words, {len(docs)} chunks)
Embedding model: sentence-transformers/all-MiniLM-L6-v2
Vector store: FAISS
LLM: llama-3.3-70b-versatile (Groq)
Evaluation: {'DeepEval FaithfulnessMetric + AnswerRelevancyMetric' if not USE_MOCK_DEEPEVAL else 'Groq LLM judge (DeepEval-style)'}
Threshold: {THRESHOLD}

Questions run: {n_total} (5 factual + 2 adversarial)
Initial pass rate: {n_initial_pass}/{n_total} ({100*n_initial_pass/n_total:.0f}%)
Final pass rate:   {n_final_pass}/{n_total} ({100*n_final_pass/n_total:.0f}%)
""")


ASSIGNMENT COMPLETE — FINAL SUMMARY

Knowledge base: Human Microbiome (~1,400 words, 44 chunks)
Embedding model: sentence-transformers/all-MiniLM-L6-v2
Vector store: FAISS
LLM: llama-3.3-70b-versatile (Groq)
Evaluation: Groq LLM judge (DeepEval-style)
Threshold: 0.7

Questions run: 5 (5 factual + 2 adversarial)
Initial pass rate: 5/5 (100%)
Final pass rate:   5/5 (100%)



## Evaluator Agent

The evaluator agent uses DeepEval metrics:
- FaithfulnessMetric
- AnswerRelevancyMetric

It evaluates the RAG output and assigns scores between 0 and 1. If both scores are above 0.7, the answer is considered PASS; otherwise, it is marked as FAIL.

## Revisor Agent

The revisor agent is triggered when the evaluator returns a FAIL verdict. It improves the answer by ensuring it is grounded in the retrieved context and addresses the issues identified by the evaluator.